# 수상작 리뷰 - 당뇨병 발병 여부 예측

## 1. 데이터

- train.csv / test.csv : 환자의 건강 검진 데이터
    - ID : 고유 ID
    - Pregnancies : 임신 횟수
    - Glucose : 포도당 농도
    - BloodPressure : 혈압
    - SkinThickness : 피부 두께
    - Insulin : 인슐린
    - BMI : 체질량지수
    - DiabetesPedigreeFunction : 당뇨병 혈통 기능
    - Age : 나이
    - Outcome : 당뇨병 여부 (Target 변수, 0: 발병되지 않음, 1: 발병)
- sample_submission.csv : 정답 파일의 예시

## 2. 코드 흐름

1. 라이브러리 설정 및 데이터 로드
    - Pandas, NumPy로 데이터 처리, Matplotlib, Seaborn으로 시각화
    - scikit-learn, XGBoost로 모델링
2. 결측치 / 이상치 처리
    - 결측치는 없지만 0으로 입력된 값을 결측치로 보고 히트맵으로 확인
    - 박스플롯으로 이상치를 확인하고, 변수별 의미를 고려해서 중앙값으로 대체
3. 데이터 분할 및 스케일링
    - train / validation을 7:3으로 나누고 StandardScaler로 표준화
4. 모델 학습
    - RandomForest, XGBoost, SVC를 묶은 Soft Voting 모델 구성
    - StratifiedKFold 5-fold 교차검증으로 ROC-AUC 확인
5. Pseudo Labeling
    - test 데이터 예측값을 정답처럼 붙여서 학습 데이터를 늘리고 재학습
6. Stacking + Threshold 튜닝
    - 같은 세 모델로 Stacking 모델 구성 (최종 모델: LogisticRegression)
    - validation에서 F1 Score가 가장 높은 임계값 탐색
7. 최종 학습 및 제출
    - train + validation 전체로 Voting 모델을 재학습하고 찾은 임계값으로 test 예측

## 3. 주요 코드 분석

**① 0값을 결측치로 보고 처리하기**

- 0이 나올 수 없는 변수(Glucose, BloodPressure, BMI)의 0은 결측치로 보고 train의 중앙값으로 대체
- 값을 그냥 지우지 않고, 의학적으로 말이 되는 값인지 따져가며 변수마다 기준을 다르게 정함
    - Insulin : 공복일 때 0일 수 있어서 0은 그대로 두고, 500 이상만 이상치로 처리
    - BMI : 50 이상은 매우 예외적인 경우라서 이상치로 처리
    - BloodPressure : 저혈압/고혈압도 당뇨와 관련 있을 수 있어서 0만 처리
- test에도 train의 중앙값을 똑같이 적용

In [ ]:
insulin = train['Insulin'].median()
train.loc[train['Insulin'] >= 500, 'Insulin'] = insulin
test.loc[test['Insulin'] >= 500, 'Insulin'] = insulin

bmi = train['BMI'].median()
train.loc[(train['BMI'] == 0) | (train['BMI'] >= 50), 'BMI'] = bmi
test.loc[(test['BMI'] == 0) | (test['BMI'] >= 50), 'BMI'] = bmi

bloodpressure = train['BloodPressure'].median()
train.loc[train['BloodPressure'] == 0, 'BloodPressure'] = bloodpressure
test.loc[test['BloodPressure'] == 0, 'BloodPressure'] = bloodpressure

glucose = train['Glucose'].median()
train.loc[train['Glucose'] == 0, 'Glucose'] = glucose
test.loc[test['Glucose'] == 0, 'Glucose'] = glucose

**② Soft Voting 앙상블**

- 성격이 다른 세 모델(RandomForest, XGBoost, SVC)의 예측 확률을 평균내서 최종 예측
- 처음엔 LogisticRegression, DecisionTree도 넣었지만 성능이 떨어져서 뺐음
- 클래스 비율을 유지하는 StratifiedKFold로 교차검증

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(random_state=42)),
        ('xgb', XGBClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cross_val_scores = cross_val_score(voting_clf, X_train_scaled, y_train, cv=kfold, scoring='roc_auc')
print("Mean cross-validation ROC-AUC:", np.mean(cross_val_scores))

voting_clf.fit(X_train_scaled, y_train)

**③ Pseudo Labeling**

- 학습한 모델로 test 데이터를 예측하고, 그 예측값을 정답처럼 붙여서 학습 데이터에 추가
- 데이터 수가 적을 때 학습 데이터를 늘리는 방법

In [ ]:
pseudo_labels = voting_clf.predict(X_test_scaled)

X_exp = np.vstack((X_train_scaled, X_test_scaled))
y_exp = np.hstack((y_train, pseudo_labels))

voting_clf.fit(X_exp, y_exp)

**④ Threshold 튜닝**

- 기본 임계값 0.5 대신, validation에서 F1 Score가 가장 높아지는 임계값을 찾아서 사용

In [ ]:
y_pred_prob = stacking_clf.predict_proba(X_val_scaled)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, y_pred_prob)

f1_scores = 2 * (precision * recall) / (precision + recall)
best_threshold = thresholds[np.argmax(f1_scores)]

y_pred_val = (y_pred_prob >= best_threshold).astype(int)

## 4. 새롭게 알게 된 점 / 아쉬운 점

**새롭게 알게 된 점**

- 결측치가 없다고 나와도 0으로 채워진 값이 사실상 결측치일 수 있다는 것을 알게 됐다. 이상치도 박스플롯 기준으로 일괄 제거하지 않고, 인슐린은 공복 시 0일 수 있는 것처럼 도메인 지식을 찾아보고 변수마다 다르게 처리하는 과정이 인상 깊었다.
- 이상치를 모두 제거해봤더니 오히려 점수가 낮아졌다는 기록을 보고, 전처리는 여러 번 실험해보면서 결정해야 한다는 것을 느꼈다.
- Pseudo Labeling, Stacking, Threshold 튜닝처럼 작은 데이터에서 성능을 끌어올리는 방법을 배웠다.

**아쉬운 점**

- 마지막 제출에는 Stacking 모델이 아니라 Voting 모델을 사용했는데, 임계값은 Stacking 모델로 찾은 값을 그대로 썼다. 모델마다 확률 분포가 달라서 같은 모델 기준으로 임계값을 찾는 게 더 맞았을 것 같다.
- SkinThickness 99는 주석에는 9의 오타로 보고 9로 바꾸자고 했지만 실제 코드는 0으로 바꿨다. 또 SkinThickness의 0은 결측치로 처리하지 않고 그대로 남아 있다.
- ROC-AUC를 확률값이 아니라 0/1 예측값으로 계산해서 실제 성능과 차이가 있을 수 있다.
- 최종 모델은 train + validation으로만 재학습해서 앞에서 한 Pseudo Labeling 효과가 최종 제출에는 반영되지 않았다.